# 维度三：数据驱动建议

综合缺口、效率和优化结果生成针对性的政策建议。


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if ROOT.name == "notebooks" and (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError("无法定位仓库根目录，请从项目根目录或 notebooks/ 目录启动 Notebook。")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)

def read_api(relpath: str):
    payload = read_json(ROOT / relpath)
    return payload.get("data", payload)

def show_image(relpath: str, width: int = 900):
    display(Image(filename=str(ROOT / relpath), width=width))

def require(*relpaths: str):
    missing = [path for path in relpaths if not (ROOT / path).exists()]
    if missing:
        raise FileNotFoundError(
            "缺少以下产物，请先运行 `PYTHONPATH=src python -m hdi.data.integrator` 和 "
            "`PYTHONPATH=src python -m hdi.analysis.competition`:\n- " + "\n- ".join(missing)
        )


In [ ]:
require(
    "api_output/dim3/resource_gap.json",
    "api_output/dim3/efficiency.json",
    "data/processed/dim2_intervention_priority.parquet",
)

gap = pd.DataFrame(read_api("api_output/dim3/resource_gap.json")).sort_values("gap").head(5)
eff = pd.DataFrame(read_api("api_output/dim3/efficiency.json"))
priority = pd.read_parquet(ROOT / "data/processed/dim2_intervention_priority.parquet")

display(Markdown("## 资源缺口最大国家"))
display(gap[["country_name", "gap_grade", "gap"]])

display(Markdown("## Q2 低投入高产出样本"))
display(eff[eff["quadrant"] == "Q2_low_input_high_output"][["country_name", "efficiency"]].sort_values("efficiency", ascending=False).head(10))

display(Markdown("## 区域主导风险与建议干预"))
display(priority[["who_region", "primary_risk", "recommended_intervention"]])
